# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [9]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V2.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-30 13:46:28.925 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

In [10]:
# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met deze logica:
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude actuele rij afsluiten + nieuwe INSERT
# - bestaande klant, niet gewijzigd -> niets doen.

# business key = klantnr uit SDM
# surrogate key = klant_sk in DWH

# We halen eerst brondata op uit beide klanttabellen.
# Dit doen we omdat Dim_Klant niet uit één bron komt, maar uit de tabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
# We voegen deze dus daarom ook verticaal samen.
query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# Bron toevoegen zodat we in logging kunnen zien waar een record vandaan komt
df_accessoire_klant["bron"] = "Accessoire_Verkoop_Klant"
df_fiets_klant["bron"] = "Fiets_Verkoop_Klant"

# Verticaal samenvoegen
df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

logger.info(f"Totaal aantal bronrecords voor Dim_Klant: {len(df_klant_source)}")

# =========================================================
# 2. DUBBELE BUSINESS KEYS CONTROLEREN
# =========================================================
# Business key = klantnr
# Als hetzelfde klantnr meerdere keren voorkomt:
# - zelfde inhoud -> 1 behouden
# - verschillende inhoud -> loggen en uitsluiten

vergelijk_kolommen = ["naam", "adres", "woonplaats", "geslacht", "geboortedatum"]

# Alle records waarvan klantnr meer dan 1x voorkomt
dubbele_klantregels = df_klant_source[
    df_klant_source.duplicated(subset=["klantnr"], keep=False)
].copy()

conflict_klantnrs = []

for klantnr, groep in dubbele_klantregels.groupby("klantnr"):
    unieke_varianten = groep[vergelijk_kolommen].astype(str).drop_duplicates()

    # Meer dan 1 unieke variant = conflict
    if len(unieke_varianten) > 1:
        conflict_klantnrs.append(klantnr)

        logger.warning(f"Conflict gevonden voor klantnr={klantnr}. Deze klant wordt NIET verwerkt in Dim_Klant.")

        for _, row in groep.iterrows():
            logger.warning(
                f"bron={row['bron']} | "
                f"klantnr={row['klantnr']} | "
                f"naam={row['naam']} | "
                f"adres={row['adres']} | "
                f"woonplaats={row['woonplaats']} | "
                f"geslacht={row['geslacht']} | "
                f"geboortedatum={row['geboortedatum']}"
            )

if not conflict_klantnrs:
    logger.info("Geen conflicterende dubbele klantnummers gevonden.")

# Conflicterende klantnummers uitsluiten
df_klant_source_clean = df_klant_source[
    ~df_klant_source["klantnr"].isin(conflict_klantnrs)
].copy()

# Gelijke dubbelen reduceren naar 1 record per klantnr
df_klant_source_clean = df_klant_source_clean.drop_duplicates(
    subset=["klantnr"]
).reset_index(drop=True)

logger.info(f"Aantal conflicten uitgesloten: {len(conflict_klantnrs)}")
logger.info(f"Aantal records na opschonen: {len(df_klant_source_clean)}")

# =========================================================
# 3. ACTUELE DWH-RECORDS OPHALEN
# =========================================================
# Alleen actuele records vergelijken: eindtijd IS NULL

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

logger.info(f"Aantal actuele klantrecords in DWH: {len(df_klant_dwh)}")

# =========================================================
# 4. WIJZIGINGSCHECK
# =========================================================

def normaliseer_waarde(value):
    if pd.isna(value):
        return None
    return str(value).strip()

def klant_is_changed(source_row, dwh_row):
    return (
        normaliseer_waarde(source_row["naam"]) != normaliseer_waarde(dwh_row["naam"]) or
        normaliseer_waarde(source_row["adres"]) != normaliseer_waarde(dwh_row["adres"]) or
        normaliseer_waarde(source_row["woonplaats"]) != normaliseer_waarde(dwh_row["woonplaats"]) or
        normaliseer_waarde(source_row["geslacht"]) != normaliseer_waarde(dwh_row["geslacht"]) or
        normaliseer_waarde(source_row["geboortedatum"]) != normaliseer_waarde(dwh_row["geboortedatum"])
    )

# =========================================================
# 5. PREVIEW TELLERS
# =========================================================

new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source_clean.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]
        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(
    f"Preview Dim_Klant | new={new_count}, changed={changed_count}, unchanged={preview_unchanged_count}"
)

# =========================================================
# 6. ETL UITVOEREN - SCD TYPE 2
# =========================================================

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source_clean.iterrows():
    klantnr = src_row["klantnr"]
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    # 1. Nieuwe klant -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1

    # 2. Bestaande klant -> check wijziging
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Oude actuele rij afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Nieuwe actuele versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}, "
    f"conflicts_excluded={len(conflict_klantnrs)}"
)

# =========================================================
# 7. RESULTAAT CONTROLEREN
# =========================================================

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, klant_sk
""", dwh_conn)

print(result_df)

2026-03-30 13:46:28.953 | INFO     | __main__:<module>:50 - Totaal aantal bronrecords voor Dim_Klant: 45
2026-03-30 13:46:28.972 | WARNING  | __main__:<module>:76 - Conflict gevonden voor klantnr=16. Deze klant wordt NIET verwerkt in Dim_Klant.
2026-03-30 13:46:28.973 | WARNING  | __main__:<module>:79 - bron=Accessoire_Verkoop_Klant | klantnr=16 | naam=Fleur Schouten | adres=Tulpenstraat 5 | woonplaats=Den Haag | geslacht=V | geboortedatum=1994-06-08
2026-03-30 13:46:28.974 | WARNING  | __main__:<module>:79 - bron=Fiets_Verkoop_Klant | klantnr=16 | naam=Robin Dekker | adres=Rozenstraat 4 | woonplaats=Delft | geslacht=M | geboortedatum=1992-11-23
2026-03-30 13:46:28.976 | WARNING  | __main__:<module>:76 - Conflict gevonden voor klantnr=17. Deze klant wordt NIET verwerkt in Dim_Klant.
2026-03-30 13:46:28.977 | WARNING  | __main__:<module>:79 - bron=Accessoire_Verkoop_Klant | klantnr=17 | naam=Lars van den Berg | adres=Kade 101 | woonplaats=Leiden | geslacht=M | geboortedatum=1988-03-02
2

    klant_sk  klantnr            naam                 adres  woonplaats  \
0         21        1      Jan Jansen         Kerkstraat 12   Amsterdam   
1         22        2  Sophie de Boer           Lindelaan 8   Rotterdam   
2         23        3   Pieter Visser         Havenstraat 3    Den Haag   
3         24        4       Emma Smit          Boomgaard 22     Haarlem   
4         25        5      Tom Bakker        Stationsweg 44      Leiden   
5         26        6     Lisa Meijer         Dijkstraat 10     Zaandam   
6         27        7   Bart de Vries      Brouwersgracht 7       Delft   
7         28        8  Julia van Dijk         Plataanlaan 5       Hoorn   
8         29        9       Kevin Mol             Singel 99     Alkmaar   
9         30       10      Nina Groen        Waterstraat 16    Schiedam   
10        31       11    Daan Willems           Parklaan 31     Haarlem   
11        32       12         Eva Vos            Zomerweg 2  Zoetermeer   
12        33       13    

Dim_Accessoire (SCD TYPE 1) ROBBERT

In [11]:
# Wat willen we bereiken?
# We willen Dim_Accessoire in het DWH vullen met SCD Type 1 logica:
# - nieuw accessoire in SDM, nog niet in DWH -> INSERT
# - bestaand accessoire, maar gewijzigd -> UPDATE (overschrijven)
# - bestaand accessoire, niet gewijzigd -> niets doen

# business key = accessoirenr uit SDM
# surrogate key = accessoire_sk in DWH

# 1. Brondata ophalen per bron (horizontale samenvoeging)
# Voor Dim_Accessoire komt de data uit:
# - Accessoire_Verkoop_Accessoire + Accessoire_Verkoop_Leverancier
# - Accessoire_Inkoop_Accessoire + Accessoire_Inkoop_Leverancier

# We JOINEN eerst per bron, omdat accessoire- en leveranciersdata
# in aparte tabellen staan.

query_accessoire_verkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Verkoop_Accessoire a
JOIN Accessoire_Verkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

query_accessoire_inkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Inkoop_Accessoire a
JOIN Accessoire_Inkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
df_accessoire_inkoop = pd.read_sql_query(query_accessoire_inkoop, sdm_conn)

logger.info(f"Aantal accessoire-rijen uit verkoopbron: {len(df_accessoire_verkoop)}")
logger.info(f"Aantal accessoire-rijen uit inkoopbron: {len(df_accessoire_inkoop)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten de rijen uit verkoop en inkoop onder elkaar.
# Daarna houden we per business key nog maar één record over.

# Combineer beide bronnen
df_all = pd.concat(
    [
        df_accessoire_verkoop.assign(bron="verkoop"),
        df_accessoire_inkoop.assign(bron="inkoop")
    ],
    ignore_index=True
)

# Groepeer per accessoirenr
df_accessoire_source = []
conflicts = []

for accessoirenr, group in df_all.groupby("accessoirenr"):
    if len(group) == 1:
        # maar 1 bron → gewoon nemen
        df_accessoire_source.append(group.iloc[0])
    else:
        # meerdere bronnen → check of ze gelijk zijn
        unieke = group.drop_duplicates(subset=[
            "naam", "soort",
            "leveranciernr", "leveranciernaam",
            "leverancieradres", "leverancierplaats"
        ])

        if len(unieke) == 1:
            df_accessoire_source.append(unieke.iloc[0])
        else:
            conflicts.append(accessoirenr)
            logger.warning(f"CONFLICT accessoirenr={accessoirenr}")
            print(group)

# maak dataframe
df_accessoire_source = pd.DataFrame(df_accessoire_source)
logger.info(f"Aantal unieke accessoires in source: {len(df_accessoire_source)}")

# 3. Actuele records uit het DWH ophalen
# We vergelijken met de huidige inhoud van Dim_Accessoire.
query_dwh_accessoire = """
SELECT
    accessoire_sk,
    accessoirenr,
    naam,
    soort,
    leveranciernr,
    leveranciernaam,
    leverancieradres,
    leverancierplaats,
    begintijd,
    eindtijd
FROM Dim_Accessoire
WHERE eindtijd IS NULL
"""

df_accessoire_dwh = pd.read_sql_query(query_dwh_accessoire, dwh_conn)

logger.info(f"Aantal actuele accessoires in DWH: {len(df_accessoire_dwh)}")

# 4. Helperfunctie: is het accessoire veranderd?
# Bij SCD Type 1 vergelijken we eerst of de inhoudelijke attributen veranderd zijn.
# Alleen dan voeren we een UPDATE uit.

def accessoire_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["soort"]) != str(dwh_row["soort"]) or
        str(source_row["leveranciernr"]) != str(dwh_row["leveranciernr"]) or
        str(source_row["leveranciernaam"]) != str(dwh_row["leveranciernaam"]) or
        str(source_row["leverancieradres"]) != str(dwh_row["leverancieradres"]) or
        str(source_row["leverancierplaats"]) != str(dwh_row["leverancierplaats"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe accessoires: {new_count}")
logger.info(f"Aantal gewijzigde accessoires: {changed_count}")
logger.info(f"Aantal ongewijzigde accessoires: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 1
# - nieuw -> INSERT
# - gewijzigd -> UPDATE
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Accessoire (SCD Type 1)")

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    # Nieuw accessoire -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr,
                naam,
                soort,
                leveranciernr,
                leveranciernaam,
                leverancieradres,
                leverancierplaats,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["accessoirenr"]),
            src_row["naam"],
            src_row["soort"],
            int(src_row["leveranciernr"]),
            src_row["leveranciernaam"],
            src_row["leverancieradres"],
            src_row["leverancierplaats"],
            now_ts
        ))
        inserted_count += 1
    # Bestaand accessoire -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            # Bij SCD Type 1 overschrijven we de bestaande rij.
            dwh_conn.execute("""
                UPDATE Dim_Accessoire
                SET
                    naam = ?,
                    soort = ?,
                    leveranciernr = ?,
                    leveranciernaam = ?,
                    leverancieradres = ?,
                    leverancierplaats = ?
                WHERE accessoire_sk = ?
            """, (
                src_row["naam"],
                src_row["soort"],
                int(src_row["leveranciernr"]),
                src_row["leveranciernaam"],
                src_row["leverancieradres"],
                src_row["leverancierplaats"],
                int(dwh_row["accessoire_sk"])
            ))
            updated_count += 1
        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Accessoire klaar | inserted={inserted_count}, "
    f"updated_scd1={updated_count}, unchanged={unchanged_count}"
)

# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Accessoire
    ORDER BY accessoirenr, accessoire_sk
""", dwh_conn)

print(result_df)

2026-03-30 13:46:29.057 | INFO     | __main__:<module>:49 - Aantal accessoire-rijen uit verkoopbron: 10
2026-03-30 13:46:29.058 | INFO     | __main__:<module>:50 - Aantal accessoire-rijen uit inkoopbron: 13
2026-03-30 13:46:29.072 | INFO     | __main__:<module>:90 - Aantal unieke accessoires in source: 13
2026-03-30 13:46:29.074 | INFO     | __main__:<module>:112 - Aantal actuele accessoires in DWH: 13
2026-03-30 13:46:29.081 | INFO     | __main__:<module>:148 - Aantal nieuwe accessoires: 0
2026-03-30 13:46:29.082 | INFO     | __main__:<module>:149 - Aantal gewijzigde accessoires: 0
2026-03-30 13:46:29.082 | INFO     | __main__:<module>:150 - Aantal ongewijzigde accessoires: 13
2026-03-30 13:46:29.083 | INFO     | __main__:<module>:163 - Start ETL voor Dim_Accessoire (SCD Type 1)
2026-03-30 13:46:29.089 | INFO     | __main__:<module>:227 - Dim_Accessoire klaar | inserted=0, updated_scd1=0, unchanged=13


    accessoire_sk  accessoirenr                      naam        soort  \
0              14             1              LED voorlamp  Verlichting   
1              15             2           LED achterlicht  Verlichting   
2              16             3  USB-oplaadbare fietslamp  Verlichting   
3              17             4              Ringslot AXA       Sloten   
4              18             5    Ketting met cijferslot       Sloten   
5              19             6         U-slot met beugel       Sloten   
6              20             7    Dubbele fietstas zwart       Tassen   
7              21             8   Enkele schouderfietstas       Tassen   
8              22             9      Waterdichte zadeltas       Tassen   
9              23            10      Kinderfietsmand roze       Tassen   
10             24            11  Reflecterende spaakclips  Verlichting   
11             25            12        Vouwbaar kabelslot       Sloten   
12             26            13       

Dim_Datum ROBBERT

In [12]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0
logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))
        inserted_count += 1

dwh_conn.commit()
logger.info(f"Dim_Datum klaar | inserted={inserted_count}")

# 7. Controle

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

2026-03-30 13:46:29.112 | INFO     | __main__:<module>:40 - Aantal datum-rijen uit Fiets_Verkoop: 150
2026-03-30 13:46:29.115 | INFO     | __main__:<module>:41 - Aantal datum-rijen uit Accessoire_Verkoop: 100
2026-03-30 13:46:29.117 | INFO     | __main__:<module>:42 - Aantal datum-rijen uit Onderhoud: 50
2026-03-30 13:46:29.122 | INFO     | __main__:<module>:55 - Aantal unieke datums in source: 196
2026-03-30 13:46:29.131 | INFO     | __main__:<module>:82 - Aantal datums in DWH: 201
2026-03-30 13:46:29.185 | INFO     | __main__:<module>:98 - Aantal nieuwe datums: 0
2026-03-30 13:46:29.187 | INFO     | __main__:<module>:99 - Aantal bestaande datums: 196
2026-03-30 13:46:29.188 | INFO     | __main__:<module>:105 - Start ETL voor Dim_Datum
2026-03-30 13:46:29.250 | INFO     | __main__:<module>:130 - Dim_Datum klaar | inserted=0


     datum_sk       datum  year  month  day
0         396  2024-01-01  2024      1    1
1         202  2024-01-02  2024      1    2
2         319  2024-01-06  2024      1    6
3         216  2024-01-08  2024      1    8
4         364  2024-01-10  2024      1   10
..        ...         ...   ...    ...  ...
196       257  2024-12-26  2024     12   26
197       379  2024-12-27  2024     12   27
198       368  2024-12-28  2024     12   28
199       291  2024-12-30  2024     12   30
200       315  2024-12-31  2024     12   31

[201 rows x 5 columns]


Fct_Verkoop ROBBERT


In [13]:
# Wat willen we bereiken?
# We willen Fct_Verkoop vullen met verkoopgebeurtenissen uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop

# In het DWH komt dit samen in één feittabel:
# Fct_Verkoop
# De feittabel bevat:
# - verkoopnr
# - verkoop_type
# - datum_sk
# - aantal
# - totaalprijs
# - standaardprijs
# - klant_sk
# - fiets_sk of accessoire_sk
# - monteur_sk

# 1. Brondata ophalen uit beide verkoopbronnen
# We voegen nu ook verkoop_type toe, zodat de samengestelde PK
# (verkoopnr, verkoop_type) correct gevuld kan worden.
logger.info("Ophalen Verkoop uit SDM")

query_fiets_verkoop = """
SELECT
    fv.fiets_verkoopnr AS verkoopnr,
    fv.datum,
    fv.aantal,
    fv.verkoopprijs AS totaalprijs,
    ff.standaardprijs,
    fv.klant,
    fv.fiets,
    fv.monteur,
    NULL AS accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff
    ON fv.fiets = ff.fietsnr
"""

query_accessoire_verkoop = """
SELECT
    av.accessoire_verkoopnr AS verkoopnr,
    av.datum,
    av.aantal,
    av.verkoopprijs AS totaalprijs,
    aa.standaardprijs,
    av.klant,
    NULL AS fiets,
    av.monteur,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa
    ON av.accessoire = aa.accessoirenr
"""

df_fiets_verkoop = pd.read_sql_query(query_fiets_verkoop, sdm_conn)
df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)

logger.info(f"Fiets verkoop: {len(df_fiets_verkoop)}")
logger.info(f"Accessoire verkoop: {len(df_accessoire_verkoop)}")

df_verkoop_source = pd.concat(
    [df_fiets_verkoop, df_accessoire_verkoop],
    ignore_index=True
)

logger.info(f"Totaal bron: {len(df_verkoop_source)}")

# --- DIMENSIES ---
df_datum_dwh = pd.read_sql_query("SELECT datum_sk, datum FROM Dim_Datum", dwh_conn)

df_klant_dwh = pd.read_sql_query("""
SELECT klant_sk, klantnr
FROM Dim_Klant
WHERE eindtijd IS NULL
""", dwh_conn)

df_fiets_dwh = pd.read_sql_query("""
SELECT fiets_sk, fietsnr
FROM Dim_Fiets
WHERE eindtijd IS NULL
""", dwh_conn)

df_accessoire_dwh = pd.read_sql_query("""
SELECT accessoire_sk, accessoirenr
FROM Dim_Accessoire
WHERE eindtijd IS NULL
""", dwh_conn)

df_monteur_dwh = pd.read_sql_query("""
SELECT monteur_sk, monteurnr
FROM Dim_Monteur
WHERE eindtijd IS NULL
""", dwh_conn)

# --- BESTAANDE RECORDS (zonder ID) ---
df_verkoop_dwh = pd.read_sql_query("""
SELECT verkoop_type, datum_sk, aantal, klant_sk
FROM Fct_Verkoop
""", dwh_conn)

bestaande_records = set(
    tuple(x) for x in df_verkoop_dwh.to_records(index=False)
)

# --- HELPERS ---
def get_datum_sk(datum):
    return int(df_datum_dwh[df_datum_dwh["datum"] == str(datum)].iloc[0]["datum_sk"])

def get_klant_sk(klantnr):
    return int(df_klant_dwh[df_klant_dwh["klantnr"] == int(klantnr)].iloc[0]["klant_sk"])

def get_fiets_sk(fietsnr):
    match = df_fiets_dwh[df_fiets_dwh["fietsnr"] == int(fietsnr)]
    return int(match.iloc[0]["fiets_sk"]) if not match.empty else None

def get_accessoire_sk(accessoirenr):
    match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == int(accessoirenr)]
    return int(match.iloc[0]["accessoire_sk"]) if not match.empty else None

def get_monteur_sk(monteurnr):
    return int(df_monteur_dwh[df_monteur_dwh["monteurnr"] == int(monteurnr)].iloc[0]["monteur_sk"])


# --- ETL ---
logger.info("Start ETL Fct_Verkoop")

inserted = 0
skipped = 0

for _, row in df_verkoop_source.iterrows():
    datum_sk = get_datum_sk(row["datum"])
    klant_sk = get_klant_sk(row["klant"])
    monteur_sk = get_monteur_sk(row["monteur"])

    # type bepalen
    if pd.notna(row["fiets"]):
        verkoop_type = "fiets"
        fiets_sk = get_fiets_sk(row["fiets"])
        accessoire_sk = None
    else:
        verkoop_type = "accessoire"
        fiets_sk = None
        accessoire_sk = get_accessoire_sk(row["accessoire"])

    # validatie
    if (fiets_sk is None and accessoire_sk is None) or \
       (fiets_sk is not None and accessoire_sk is not None):
        logger.warning(f"Conflict in verkoop: {row}")
        continue

    record_key = (
        verkoop_type,
        datum_sk,
        int(row["aantal"]),
        klant_sk
    )

    if record_key in bestaande_records:
        skipped += 1
        continue

    dwh_conn.execute("""
        INSERT INTO Fct_Verkoop (
            verkoop_type,
            datum_sk,
            aantal,
            totaalprijs,
            standaardprijs,
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        verkoop_type,
        datum_sk,
        int(row["aantal"]),
        float(row["totaalprijs"]),
        float(row["standaardprijs"]),
        klant_sk,
        fiets_sk,
        accessoire_sk,
        monteur_sk
    ))

    inserted += 1

dwh_conn.commit()

logger.info(f"Fct_Verkoop klaar inserted={inserted}, skipped={skipped}")

2026-03-30 13:46:29.271 | INFO     | __main__:<module>:22 - Ophalen Verkoop uit SDM
2026-03-30 13:46:29.275 | INFO     | __main__:<module>:59 - Fiets verkoop: 150
2026-03-30 13:46:29.276 | INFO     | __main__:<module>:60 - Accessoire verkoop: 100
2026-03-30 13:46:29.278 | INFO     | __main__:<module>:67 - Totaal bron: 250
2026-03-30 13:46:29.283 | INFO     | __main__:<module>:126 - Start ETL Fct_Verkoop


IndexError: single positional indexer is out-of-bounds

Dim_Fiets (SCD Type 2) MEES

In [ ]:
logger.info("Ophalen Fiets uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, naam, adres, plaats
FROM (
    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Inkoop_Fiets f
    JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Verkoop_Fiets f
    JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Onderhoud_Fiets f
    JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
)
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
    fiets_sk
FROM Dim_Fiets
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

for row in source_data:
    (fietsnr, soort, merk, type_, kleur,
     fabrikantnr, fab_naam, fab_adres, fab_plaats) = row

    now = datetime.now()

    if fietsnr not in dwh_data:
        # nieuw
        logger.info(f"Nieuwe fiets: {fietsnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (fietsnr, soort, merk, type_, kleur,
              fabrikantnr, fab_naam, fab_adres, fab_plaats,
              now))

    else:
        # check wijzigingen
        dwh_row = dwh_data[fietsnr]

        (_, d_soort, d_merk, d_type, d_kleur,
         d_fabnr, d_fabnaam, d_fabadres, d_fabplaats,
         fiets_sk) = dwh_row

        if (soort, merk, type_, kleur,
            fabrikantnr, fab_naam, fab_adres, fab_plaats) != \
           (d_soort, d_merk, d_type, d_kleur,
            d_fabnr, d_fabnaam, d_fabadres, d_fabplaats):

            logger.info(f"Update in fiets: {fietsnr}")

            # 1. Sluit oude (voeg eindtijd toe)
            dwh_cursor.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, fiets_sk))

            # 2. Maak nieuwe (met nieuwe gegevens en begintijd)
            dwh_cursor.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (fietsnr, soort, merk, type_, kleur,
                  fabrikantnr, fab_naam, fab_adres, fab_plaats,
                  now))

dwh_conn.commit()
logger.info("Fiets dimensie geupdate in DWH.")

Dim_Monteur (SCD Type 1) MEES

In [ ]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr, naam, woonplaats, uurloon,
    filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (monteurnr, naam, woonplaats, uurloon,
     filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()

    if monteurnr not in dwh_data:
        # nieuwe regels worden toegevoegd
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr, naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (monteurnr, naam, woonplaats, uurloon,
              filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
              now))

    else:
        # bestaande regels worden overschreven als er wijzigingen zijn
        (_, d_naam, d_woonplaats, d_uurloon,
         d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie,
         monteur_sk) = dwh_data[monteurnr]

        if (naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) != \
           (d_naam, d_woonplaats, d_uurloon,
            d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?, woonplaats = ?, uurloon = ?,
                filiaalnr = ?, filiaalnaam = ?, filiaaladres = ?, filiaalprovincie = ?
            WHERE monteur_sk = ?
            """, (naam, woonplaats, uurloon,
                  filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
                  monteur_sk))
            
dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

Fct_Onderhoud MEES

In [ ]:
logger.info("Ophalen Onderhoud uit SDM")

sdm_cursor.execute("""
SELECT 
    o.onderhoudnr,
    o.datum,
    o.starttijd,
    o.eindtijd,
    o.fiets,
    o.monteur
FROM Onderhoud o
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT onderhoudnr FROM Fct_Onderhoud")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_or_create_datum_sk(datum):
    dwh_cursor.execute("SELECT datum_sk FROM Dim_Datum WHERE datum = ?", (datum,))
    if r := dwh_cursor.fetchone():
        return r[0]

    from datetime import datetime
    d = datetime.strptime(datum, "%Y-%m-%d")

    dwh_cursor.execute("""
    INSERT INTO Dim_Datum (datum, year, month, day)
    VALUES (?, ?, ?, ?)
    """, (datum, d.year, d.month, d.day))

    return dwh_cursor.lastrowid


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None


logger.info("Start ETL Fct_Onderhoud")

for row in source_data:
    onderhoudnr, datum, starttijd, eindtijd, fietsnr, monteurnr = row

    # skip bestaande
    if onderhoudnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    monteur_sk = get_monteur_sk(monteurnr)

    # Check of alle keys bestaan
    if not all([datum_sk, fiets_sk, monteur_sk]):
        logger.warning(f"SK ontbreekt voor onderhoud {onderhoudnr}")
        continue

    # Insert fact
    dwh_cursor.execute("""
    INSERT INTO Fct_Onderhoud (
        onderhoudnr,
        datum_sk,
        starttijd,
        eindtijd,
        fiets_sk,
        monteur_sk
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (onderhoudnr, datum_sk, starttijd, eindtijd, fiets_sk, monteur_sk))

dwh_conn.commit()
logger.success("Fct_Onderhoud load klaar")

Fct_Inkoop MEES

In [ ]:
logger.info("Ophalen Inkoop uit SDM") 

sdm_cursor.execute("""
SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    f.inkoopprijs,
    i.fiets,
    NULL as accessoire
FROM Fiets_Inkoop i
JOIN Fiets_Inkoop_Fiets f ON i.fiets = f.fietsnr

UNION ALL

SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    a.inkoopprijs,
    NULL as fiets,
    i.accessoire
FROM Accessoire_Inkoop i
JOIN Accessoire_Inkoop_Accessoire a ON i.accessoire = a.accessoirenr
""")

source_data = sdm_cursor.fetchall()

# LET OP: we gebruiken nu de SOURCE inkoopnr alleen voor incremental check
dwh_cursor.execute("SELECT inkoop_type, datum_sk, aantal FROM Fct_Inkoop")
bestaande_records = set(dwh_cursor.fetchall())


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


logger.info("Start ETL Fct_Inkoop")

for row in source_data:
    source_inkoopnr, datum, aantal, inkoopprijs, fietsnr, accessoirenr = row

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    accessoire_sk = get_accessoire_sk(accessoirenr)

    # bepaal type
    if fiets_sk is not None:
        inkoop_type = "fiets"
    elif accessoire_sk is not None:
        inkoop_type = "accessoire"
    else:
        logger.warning(f"Ongeldige inkoop (geen match): {source_inkoopnr}")
        continue

    # validatie (extra check)
    if fiets_sk is not None and accessoire_sk is not None:
        logger.warning(f"Dubbele koppeling bij {source_inkoopnr}")
        continue

    totaalprijs = aantal * inkoopprijs

    # incremental check (zonder auto-id)
    record_key = (inkoop_type, datum_sk, aantal)
    if record_key in bestaande_records:
        continue

    dwh_cursor.execute("""
    INSERT INTO Fct_Inkoop (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    ))

dwh_conn.commit()
logger.success("Fct_Inkoop load klaar")